## 1. Importar librerías

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
import pandas as pd
import numpy as np
import tensorflow as tf
import sklearn
import pickle
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

## 2. Cargar y preparar el dataset

Columnas de entrada: `nombre_tienda`, `subcategoria`, `esencial`
Columna objetivo: `categoria_principal`

Se descartan `id_cliente` (identificador, sin señal de categoría), `monto` (valor único por fila,
no correlacionado categóricamente según el EDA) y `metodo_pago` (baja correlación con el target
según el EDA).

In [ ]:
def normalizar_esencial(serie):
    """Convierte la columna 'esencial' (si/no, true/false, 1/0) a float 0/1."""
    serie = serie.astype(str).str.strip().str.lower()
    mapa = {'si': 1, 'sí': 1, 'true': 1, '1': 1, 'no': 0, 'false': 0, '0': 0}
    return serie.map(mapa).astype('float32')

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/b/c/dataset.csv")

X_nombre = df['nombre_tienda'].astype(str).values
X_subcategoria = df['subcategoria'].astype(str).values
X_esencial = normalizar_esencial(df['esencial']).values.reshape(-1, 1)

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['categoria_principal'])
num_clases = len(label_encoder.classes_)

print(f"Clases categoria_principal: {list(label_encoder.classes_)}")

(X_nom_train, X_nom_test,
 X_sub_train, X_sub_test,
 X_esc_train, X_esc_test,
 y_train, y_test) = train_test_split(
    X_nombre, X_subcategoria, X_esencial, y, test_size=0.2, random_state=42
)

## 3. Vectorización de texto

In [ ]:
max_tokens = 5000
sequence_length = 5

vectorize_layer = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=sequence_length
)

# Se ajusta (adapt) solo con datos de train para evitar fuga de información (data leakage)
vectorize_layer.adapt(np.concatenate((X_nom_train, X_sub_train)))

## 4. Bloque Transformer (encoder)

In [ ]:
def transformer_encoder(inputs, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
    attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)(inputs, inputs)
    attn_output = layers.Dropout(dropout_rate)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(inputs + attn_output)

    ffn_output = layers.Dense(ff_dim, activation="relu")(out1)
    ffn_output = layers.Dense(embed_dim)(ffn_output)
    ffn_output = layers.Dropout(dropout_rate)(ffn_output)
    return layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output)

embed_dim = 64
num_heads = 2
ff_dim = 64

## 5. Modelo — `nombre_tienda` + `subcategoria` + `esencial`

In [ ]:
input_nombre = layers.Input(shape=(1,), dtype=tf.string, name='input_nombre')
input_subcategoria = layers.Input(shape=(1,), dtype=tf.string, name='input_subcategoria')
input_esencial = layers.Input(shape=(1,), dtype=tf.float32, name='input_esencial')

emb_nombre = layers.Embedding(input_dim=max_tokens, output_dim=embed_dim)(vectorize_layer(input_nombre))
emb_subcategoria = layers.Embedding(input_dim=max_tokens, output_dim=embed_dim)(vectorize_layer(input_subcategoria))

trans_nombre = transformer_encoder(emb_nombre, embed_dim, num_heads, ff_dim)
trans_subcategoria = transformer_encoder(emb_subcategoria, embed_dim, num_heads, ff_dim)

pool_nombre = layers.GlobalAveragePooling1D()(trans_nombre)
pool_subcategoria = layers.GlobalAveragePooling1D()(trans_subcategoria)

context_fusion = layers.Concatenate(name='context_fusion')([pool_nombre, pool_subcategoria, input_esencial])
context_fusion = layers.Dense(128, activation='relu')(context_fusion)
context_fusion = layers.Dropout(0.2)(context_fusion)

output_categoria = layers.Dense(num_clases, activation='softmax', name='salida_categoria')(context_fusion)

model_full = Model(
    inputs=[input_nombre, input_subcategoria, input_esencial],
    outputs=output_categoria,
    name='modelo_full'
)

model_full.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_full.summary()

In [ ]:
history_full = model_full.fit(
    {'input_nombre': X_nom_train, 'input_subcategoria': X_sub_train, 'input_esencial': X_esc_train},
    y_train,
    validation_data=(
        {'input_nombre': X_nom_test, 'input_subcategoria': X_sub_test, 'input_esencial': X_esc_test},
        y_test
    ),
    epochs=10,
    batch_size=32
)

## 6. Evaluación — Precision / Recall / F1 por clase

In [ ]:
pred_probs = model_full.predict(
    {'input_nombre': X_nom_test, 'input_subcategoria': X_sub_test, 'input_esencial': X_esc_test},
    verbose=0
)
y_pred = np.argmax(pred_probs, axis=1)

acc_full = history_full.history['val_accuracy'][-1]
print(f"Accuracy validación (última época): {acc_full:.2%}\n")

reporte = classification_report(
    y_test, y_pred, target_names=label_encoder.classes_, digits=3
)
print("Precision / Recall / F1-score por clase (sobre el set de test):\n")
print(reporte)

## 7. Guardar modelo y artefactos

In [ ]:
model_full.save("modelo_categoria_full.keras")

config_vectorizador = {
    'vocabulario': vectorize_layer.get_vocabulary()
}

try:
    import keras
    keras_version = keras.__version__
except ImportError:
    keras_version = tf.__version__

versiones_librerias = {
    'python': sys.version.split()[0],
    'tensorflow': tf.__version__,
    'keras': keras_version,
    'scikit-learn': sklearn.__version__,
    'numpy': np.__version__,
    'pandas': pd.__version__
}

artefactos = {
    'label_encoder': label_encoder,
    'config_vectorizador': config_vectorizador,
    'versiones_librerias': versiones_librerias
}

with open('artefactos_categoria.pkl', 'wb') as f:
    pickle.dump(artefactos, f)

print("Versiones de librerías utilizadas:")
for lib, v in versiones_librerias.items():
    print(f"  {lib}: {v}")